# Payphone Full Import – Merge + GGUF on Colab T4

Merge your trained LoRA adapter and convert to GGUF entirely on Colab. Download a single **payphone-story.gguf** file. No heavy work on your machine.

**Before running:** Runtime → Change runtime type → **T4 GPU**  
**If OOM:** Runtime → Restart session, then run all from top (fresh GPU).

## 1. Install dependencies

In [ ]:
# Reduce CUDA fragmentation (run first; restart session if you had OOM)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q --upgrade pyarrow
!pip install -q unsloth transformers peft bitsandbytes accelerate

## 2. Upload LoRA adapter

Upload `payphone-storyteller-lora.zip` (from Colab training download).

In [ ]:
from google.colab import files
import zipfile
import os

print("Upload payphone-storyteller-lora.zip")
uploaded = files.upload()
zip_path = list(uploaded.keys())[0] if uploaded else None
if not zip_path:
    raise FileNotFoundError("Upload payphone-storyteller-lora.zip")

os.makedirs("/content/lora", exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/lora")
ADAPTER_PATH = "/content/lora/payphone-storyteller-lora"
if not os.path.exists(ADAPTER_PATH):
    ADAPTER_PATH = "/content/lora"
print(f"Adapter at: {ADAPTER_PATH}")

## 3. Merge LoRA into base model

In [ ]:
import unsloth  # Must be before transformers, peft
from unsloth import FastLanguageModel
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
from peft import PeftModel

BASE = "Qwen/Qwen2.5-7B-Instruct"
GGUF_DIR = "/content/gguf_output"
import os
os.makedirs(GGUF_DIR, exist_ok=True)

print("Loading base model (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE,
    max_seq_length=512,
    load_in_4bit=True,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)

print("Merging...")
model = model.merge_and_unload()

print("Saving to GGUF (q8_0) via Unsloth - handles bitsandbytes...")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q8_0")
print("Done.")

## 4. Download GGUF

In [ ]:
from google.colab import files
import glob
import shutil
gguf_files = glob.glob("/content/gguf_output/*.gguf")
if gguf_files:
    shutil.copy(gguf_files[0], "/content/payphone-story.gguf")
    files.download("/content/payphone-story.gguf")
    print("Download started. Place in project root, then: ollama create payphone-story -f Modelfile")
else:
    print("ERROR: GGUF not found. Check Cell 3 for errors.")

## Done

In [ ]:
# GGUF created by Unsloth in Cell 3. Download in Cell 4 above.